# Part 1 - From Script to the Shared Computing Cluster (SCC)

So far, everything you've written has run on your **local computer**. For the rest of the workshop (Part 2), you'll run Python on the **Shared Computing Cluster (SCC)** - Boston University's Research Computing Services cluster, which also gives you access to GPUs.

This notebook introduces the basic workflow for taking a Python script you wrote locally and submitting it as a **batch job** on the SCC. The SCC project you'll use for this workshop is *nsf-energize*.

## Step 1: Turn Your Notebook Code into a `.py` Script

A Jupyter notebook is great for exploring data interactively. If you want to convert a notebook code to a script:

1. Combine your code cells, in order, into a single `.py` file.
2. Wrap the main logic in a `main()` function.
3. Add the `if __name__ == "__main__": main()` block (see Notebook 4).
4. Replace anything that requires a live display (like `plt.show()`) with saving results to a file, e.g. `plt.savefig("plot.png")`.
5. Remove or guard any `!shell` / `%magic` commands - those only work in Jupyter.

### Example: A Notebook Cell...

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

strain = np.linspace(0, 0.01, 50)
stress = 50000 * strain * np.exp(-50 * strain)

plt.plot(strain, stress)
plt.xlabel("Strain")
plt.ylabel("Stress (MPa)")
plt.title("Simulated Stress-Strain Curve")
plt.show()

### ...Becomes a Script (`stress_strain.py`)

We use `%%writefile` to save this cell as a standalone script, exactly what you would submit to the SCC:

In [ ]:
%%writefile stress_strain.py
import numpy as np
import matplotlib
matplotlib.use("Agg")  # No display available on a compute node - use a non-interactive backend
import matplotlib.pyplot as plt

def simulate_stress_strain():
    """Generates a simulated stress-strain curve and saves a plot."""
    strain = np.linspace(0, 0.01, 50)
    stress = 50000 * strain * np.exp(-50 * strain)

    plt.plot(strain, stress)
    plt.xlabel("Strain")
    plt.ylabel("Stress (MPa)")
    plt.title("Simulated Stress-Strain Curve")
    plt.savefig("stress_strain.png")
    print("Saved plot to stress_strain.png")

def main():
    simulate_stress_strain()

if __name__ == "__main__":
    main()

In [ ]:
# Run it locally, just like it would run on a compute node
!python stress_strain.py

## Step 2: Logging into the SCC


You can access the SCC through a web browser using **SCC OnDemand** at [scc-ondemand.bu.edu](scc-ondemand.bu.edu), which gives you a file browser, a terminal, and the ability to launch Jupyter notebooks directly on a compute node (more on this in Part 2).

You can also access the SCC through SSH. From a terminal (Mac/Linux) or an SSH client such as PuTTY (Windows):

```bash
ssh your_bu_username@scc1.bu.edu
```

You'll be prompted for your BU Kerberos password and Duo two-factor authentication. Once logged in, you land on a **login node** - used for editing files and submitting jobs, but **not** for running heavy computations directly.

## Step 3: Loading the Python Module

The SCC uses a **module system** to manage software versions. Before running Python, load a module:

```bash
module load python3/3.13.8
```

You can see what versions are available with:

```bash
module avail python3
```

And check which modules you currently have loaded with:

```bash
module list
```

You can also use the **miniconda** module to create a virtual environment with your own packages. For example, to create a new environment called `energize`:

```bash
module load miniconda
setup_scc_condarc.sh  # run it only once to set up your conda configuration
conda create --name energize python=3.13 notebook numpy pandas matplotlib
conda activate energize
```

## Step 4: Submitting a Batch Job with `qsub`

The SCC uses the **Sun Grid Engine (SGE)** batch scheduler. Instead of running your script directly on the login node, you submit it as a **job**, and the scheduler runs it on an available compute node.

A batch job script is a shell script with special `#$` comment lines that tell the scheduler what resources you need. Here is a minimal example, `run_stress_strain.qsub`:

```bash
#!/bin/bash -l

#$ -N stress_strain      # Job name
#$ -l h_rt=00:10:00       # Maximum run time (hh:mm:ss)
#$ -j y                   # Merge error and output logs

module load minimconda
conda activate energize
python stress_strain.py
```

You submit this script with:

```bash
qsub run_stress_strain.qsub
```

## Step 5: Monitoring Your Job

- `qstat -u your_username` - shows the status of your jobs (queued, running, or finished).
- Once the job finishes, look for an output file named `stress_strain.o<job_id>` in the directory where you submitted the job - it contains anything your script printed.
- Any files your script creates (like `stress_strain.png`) will appear in that same directory, and you can download them or view them through SCC OnDemand's file browser.

### *Exercise*
1. Save one of your own notebook cells from earlier today (e.g., the density calculator function, or the NumPy heat-treatment exercise) as a `.py` script using the pattern shown above.
2. Write a matching `.qsub` batch script for it.

We'll actually log into the SCC and submit jobs together in **Part 2**, where we'll also cover requesting GPU nodes and setting up virtual environments.